In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("GPU available:", torch.cuda.is_available())

mean = (0.4914, 0.4822, 0.4465)
std = (0.2470, 0.2435, 0.2616)

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=False)

class ResNet18_CIFAR(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        base = torchvision.models.resnet18(weights=None, num_classes=num_classes)
        base.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        base.maxpool = nn.Identity()
        self.model = base
    def forward(self, x):
        return self.model(x)

criterion = nn.CrossEntropyLoss()
print("Setup ready")

GPU available: True


100%|██████████| 170M/170M [17:21<00:00, 164kB/s]  


Setup ready


In [2]:
def fgsm_attack_train(model, images, labels, epsilon):
    images = images.clone().detach().to(device)
    images.requires_grad = True
    outputs = model(images)
    loss = criterion(outputs, labels)
    model.zero_grad()
    loss.backward()
    perturbed = images + epsilon * images.grad.sign()
    min_vals = ((0 - torch.tensor(mean)) / torch.tensor(std)).view(1,3,1,1).to(images.device)
    max_vals = ((1 - torch.tensor(mean)) / torch.tensor(std)).view(1,3,1,1).to(images.device)
    perturbed = torch.max(torch.min(perturbed, max_vals), min_vals)
    return perturbed.detach()

epsilon = 0.03
model_fgsm_only = ResNet18_CIFAR().to(device)
optimizer_fgsm_only = optim.Adam(model_fgsm_only.parameters(), lr=0.001)

for epoch in range(20):
    model_fgsm_only.train()
    running_loss = 0.0
    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)
        adv_images = fgsm_attack_train(model_fgsm_only, images, labels, epsilon)
        combined_images = torch.cat([images, adv_images], dim=0)
        combined_labels = torch.cat([labels, labels], dim=0)
        optimizer_fgsm_only.zero_grad()
        outputs = model_fgsm_only(combined_images)
        loss = criterion(outputs, combined_labels)
        loss.backward()
        optimizer_fgsm_only.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}/20, Loss: {running_loss/len(trainloader):.4f}")

Epoch 1/20, Loss: 1.6095
Epoch 2/20, Loss: 1.2190
Epoch 3/20, Loss: 1.0216
Epoch 4/20, Loss: 0.8999
Epoch 5/20, Loss: 0.8131
Epoch 6/20, Loss: 0.7462
Epoch 7/20, Loss: 0.6990
Epoch 8/20, Loss: 0.6460
Epoch 9/20, Loss: 0.6079
Epoch 10/20, Loss: 0.5752
Epoch 11/20, Loss: 0.5393
Epoch 12/20, Loss: 0.5135
Epoch 13/20, Loss: 0.4835
Epoch 14/20, Loss: 0.4612
Epoch 15/20, Loss: 0.4376
Epoch 16/20, Loss: 0.4107
Epoch 17/20, Loss: 0.3882
Epoch 18/20, Loss: 0.3679
Epoch 19/20, Loss: 0.3500
Epoch 20/20, Loss: 0.3296


In [3]:
def fgsm_attack(image, epsilon, data_grad):
    sign_data_grad = data_grad.sign()
    perturbed_image = image + epsilon * sign_data_grad
    min_vals = ((0 - torch.tensor(mean)) / torch.tensor(std)).view(1,3,1,1).to(image.device)
    max_vals = ((1 - torch.tensor(mean)) / torch.tensor(std)).view(1,3,1,1).to(image.device)
    perturbed_image = torch.max(torch.min(perturbed_image, max_vals), min_vals)
    return perturbed_image

def pgd_attack(model, images, labels, epsilon, alpha, iters):
    min_vals = ((0 - torch.tensor(mean)) / torch.tensor(std)).view(1,3,1,1).to(images.device)
    max_vals = ((1 - torch.tensor(mean)) / torch.tensor(std)).view(1,3,1,1).to(images.device)
    original_images = images.clone().detach()
    perturbed = original_images + torch.empty_like(original_images).uniform_(-epsilon, epsilon)
    perturbed = torch.max(torch.min(perturbed, max_vals), min_vals)
    for i in range(iters):
        perturbed.requires_grad = True
        outputs = model(perturbed)
        loss = criterion(outputs, labels)
        model.zero_grad()
        loss.backward()
        data_grad = perturbed.grad.data
        perturbed = perturbed.detach() + alpha * data_grad.sign()
        eta = torch.clamp(perturbed - original_images, min=-epsilon, max=epsilon)
        perturbed = (original_images + eta).detach()
        perturbed = torch.max(torch.min(perturbed, max_vals), min_vals)
    return perturbed

epsilon = 0.03
alpha = 0.007

model_fgsm_only.eval()


correct = total = 0
with torch.no_grad():
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model_fgsm_only(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
baseline_acc = 100 * correct / total
print(f"Baseline (clean): {baseline_acc:.2f}%")

# FGSM
correct_fgsm = total_fgsm = 0
for images, labels in testloader:
    images, labels = images.to(device), labels.to(device)
    images.requires_grad = True
    outputs = model_fgsm_only(images)
    loss = criterion(outputs, labels)
    model_fgsm_only.zero_grad()
    loss.backward()
    data_grad = images.grad.data
    perturbed_images = fgsm_attack(images, epsilon, data_grad)
    outputs_attacked = model_fgsm_only(perturbed_images)
    _, predicted_attacked = torch.max(outputs_attacked, 1)
    total_fgsm += labels.size(0)
    correct_fgsm += (predicted_attacked == labels).sum().item()
fgsm_acc = 100 * correct_fgsm / total_fgsm
print(f"FGSM attack: {fgsm_acc:.2f}%")


for pgd_steps in [5, 10, 20, 40]:
    correct_pgd = total_pgd = 0
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        perturbed = pgd_attack(model_fgsm_only, images, labels, epsilon, alpha, pgd_steps)
        outputs = model_fgsm_only(perturbed)
        _, predicted = torch.max(outputs, 1)
        total_pgd += labels.size(0)
        correct_pgd += (predicted == labels).sum().item()
    pgd_acc = 100 * correct_pgd / total_pgd
    print(f"PGD-{pgd_steps} attack: {pgd_acc:.2f}%")

Baseline (clean): 88.85%
FGSM attack: 73.08%
PGD-5 attack: 75.66%
PGD-10 attack: 71.73%
PGD-20 attack: 71.46%
PGD-40 attack: 71.41%
